# Step: Data Cleaning
## Al-Rajab Lenders — Borrower Default Risk Analysis

**Purpose:** Apply data quality decisions to produce a clean, analysis-ready dataset for credit
risk modeling.

**Source:** `cs-training.csv` (Kaggle "Give Me Some Credit" competition dataset, 150,000 rows)

**Decisions being applied in this notebook:**
1. Remove the single row with `age = 0` (a data entry error — not a valid borrower age)
2. Impute missing `MonthlyIncome` values with the median, and add an `Income_Was_Missing` flag
   column to preserve the missingness signal
3. Impute missing `NumberOfDependents` values with the median (which is 0, based on the real
   distribution — most borrowers have no dependents)
4. Keep `MonthlyIncome = 0` rows as-is (verified as likely genuine, not an error — these rows
   actually show a *lower* default rate than the rest of the dataset)
5. Run final validation checks before saving the cleaned dataset


## 1. Import Libraries and Load Data

In [1]:
import pandas as pd
import numpy as np

# Load raw dataset (first column is an unnamed row index from the original export)
df = pd.read_csv('cs-training.csv', index_col=0)

print(f"Initial shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


Initial shape: 150000 rows, 11 columns


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


## 2. Remove Invalid Age Row

Only 1 row out of 150,000 has `age = 0`, which is not a valid borrower age — almost certainly a
data entry error. Since this affects a single row (not the whole column), we drop just this row
rather than dropping the `age` column, which remains a potentially valuable predictor.


In [2]:
before = df.shape[0]
df = df[df['age'] > 0].copy()
after = df.shape[0]

print(f"Rows before removing invalid age: {before}")
print(f"Rows removed: {before - after}")
print(f"Rows remaining: {after}")


Rows before removing invalid age: 150000
Rows removed: 1
Rows remaining: 149999


## 3. Handle Missing `MonthlyIncome`

`MonthlyIncome` is missing in ~19.8% of records. Since this is a continuous numeric column (not
categorical), missing values cannot simply be labeled "Unknown" — doing so would break any
numeric calculation or model. Instead:

- A new flag column `Income_Was_Missing` is created (1 = was missing, 0 = was present), preserving
  the missingness signal for later analysis.
- The actual missing values are filled with the **median** (not the mean), since income is
  right-skewed with extreme outliers (max value seen: ~3,000,000/month) that would distort a
  mean-based fill.

Note: `MonthlyIncome = 0` rows are NOT touched here — verification showed these likely represent
genuine zero-income borrowers (e.g. retired, or reliant on non-employment income), with a *lower*
observed default rate than the rest of the dataset, not a data error.


In [3]:
# Flag rows where income was missing, before imputing
df['Income_Was_Missing'] = df['MonthlyIncome'].isnull().astype(int)

income_median = df['MonthlyIncome'].median()
print(f"MonthlyIncome median (used for imputation): {income_median}")

df['MonthlyIncome'] = df['MonthlyIncome'].fillna(income_median)

print(f"\nRemaining missing MonthlyIncome values: {df['MonthlyIncome'].isnull().sum()}")
print(f"Rows flagged as Income_Was_Missing: {df['Income_Was_Missing'].sum()}")


MonthlyIncome median (used for imputation): 5400.0

Remaining missing MonthlyIncome values: 0
Rows flagged as Income_Was_Missing: 29731


## 4. Handle Missing `NumberOfDependents`

`NumberOfDependents` is missing in ~2.6% of records — and every one of these rows also has a
missing `MonthlyIncome` (confirmed: 3,924 missing in both, exactly overlapping), suggesting a
group of borrowers with generally incomplete application information.

The real distribution of `NumberOfDependents` shows most borrowers (58%) have 0 dependents, with
a median and mode of 0 — not 3-5 as might be intuitively assumed. Missing values are imputed with
this median (0).


In [4]:
dependents_median = df['NumberOfDependents'].median()
print(f"NumberOfDependents median (used for imputation): {dependents_median}")

df['NumberOfDependents'] = df['NumberOfDependents'].fillna(dependents_median)

print(f"\nRemaining missing NumberOfDependents values: {df['NumberOfDependents'].isnull().sum()}")


NumberOfDependents median (used for imputation): 0.0

Remaining missing NumberOfDependents values: 0


## 5. Final Validation Checks

Before saving the cleaned dataset, confirm:
- No remaining missing values anywhere in the dataset
- Final row and column counts make sense given the cleaning steps applied
- Target variable distribution is unchanged (class imbalance should still be ~6.7%)


In [5]:
# Check for any remaining missing values
remaining_missing = df.isnull().sum().sum()
print(f"Remaining missing values in dataset: {remaining_missing}")

# Confirm target variable distribution is stable
print(f"\nTarget variable (SeriousDlqin2yrs) distribution:")
print(df['SeriousDlqin2yrs'].value_counts(normalize=True) * 100)

# Final shape summary
print(f"\nFinal cleaned dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")


Remaining missing values in dataset: 0

Target variable (SeriousDlqin2yrs) distribution:
SeriousDlqin2yrs
0    93.315955
1     6.684045
Name: proportion, dtype: float64

Final cleaned dataset shape: 149999 rows, 12 columns


## 6. Save Cleaned Dataset

Save the cleaned dataset as a new file, keeping the original raw file untouched — same
reproducibility practice as the previous project.


In [6]:
df.to_csv('credit_risk_cleaned.csv', index=False)
print("Cleaned dataset saved as 'credit_risk_cleaned.csv'")


Cleaned dataset saved as 'credit_risk_cleaned.csv'


## Summary of Cleaning Actions

| Action | Detail |
|---|---|
| Rows removed | 1 row with invalid `age = 0` |
| Columns added | `Income_Was_Missing` (missingness flag) |
| Columns imputed | `MonthlyIncome` (median), `NumberOfDependents` (median = 0) |
| Columns unchanged | `MonthlyIncome = 0` rows kept as-is (verified genuine) |
| Output | `credit_risk_cleaned.csv` |

**Next step:** Exploratory Data Analysis (EDA) on the cleaned dataset.
